# Analiza Porównawcza i Walidacja Metod Redukcji Wymiarowości (Bełchatów)

Poniższe zestawienie przedstawia zaawansowaną analizę porównawczą trzech podejść do ekstrakcji cech spektralnych dla obszaru kopalni odkrywkowej w Bełchatówie na podstawie danych satelitarnych **Sentinel-2**. Eksperyment obrazuje ewolucję przetwarzania danych: od klasycznej analizy kowariancji, przez autorski rozkład SVD ze standaryzacją, aż po ukierunkowaną filtrację sygnału albedo.

---

### 1. Analiza Statystyczna Wyjaśnionej Wariancji (Scree Plots)

Decomposition macierzowa pozwoliła na precyzyjne określenie struktury informacji zakodowanej w poszczególnych składowych głównych (PC):

* **Zwykłe PCA (Maksymalizacja Kowariancji):** Pierwsza składowa zdominowała zbiór, skupiając aż **55.39%** wariancji. Całkowita informacja niesiona przez pierwsze 3 składowe wynosi **97.26%**. 
* **Autorskie SVD z Z-score (Standaryzacja Skali):** Wprowadzenie normalizacji sprowadziło pasma do wspólnej jednostki. Udział $PC1$ spadł do **53.21%**, podczas gdy informacja w $PC2$ wzrosła do **41.60%**. Trzy składowe tłumaczą łącznie **96.67%** zmienności pierwotnej.
* **Eksperyment Antka (Filtracja Spektralna):** Świadome odrzucenie składowej $PC1$ zredukowało całkowitą wariancję kompozycji do **44.92%** (suma $PC2-PC4$). Pozwoliło to jednak na całkowite usunięcie dominującego szumu jasności tła.

---

### 2. Interpretacja Fizyczna i Teledetekcyjna Obrazów

Każda z wygenerowanych kompozycji RGB charakteryzuje się odmienną czułością na obiekty fizyczne znajdujące się na powierzchni ziemi:

#### Panel 1: Zwykłe PCA (PC1, PC2, PC3)
* **Charakterystyka:** Dominacja surowej jasności (albedo). Algorytm reaguje głównie na pasma podczerwone o najwyższej amplitudzie sygnału.
* **Wizualizacja terenu:** Wyrobisko kopalni odcina się jako jaskrawa, seledynowo-żółta struktura, tracąc jednak wewnętrzne szczegóły geologiczne. Otaczająca szata roślinna oraz struktura pól rolniczych ulegają silnemu spłaszczeniu spektralnemu (jednolite ciemne tło).

#### Panel 2: Autorskie SVD z Z-score (PC1, PC2, PC3)
* **Charakterystyka:** Optymalne zbalansowanie wag spektralnych. Każde pasmo satelitarne (od widzialnych po SWIR) ma równy wpływ na kierunki wektorów osobliwych.
* **Wizualizacja terenu:** Uzyskano maksymalną separację obiektów. Kopalnia zyskuje intensywny, krwistoczerwony odcień, ujawniając układ tarasów eksploatacyjnych. Otaczające lasy (głęboki granat) oraz mozaika pól (błękity i róże) są doskonale odseparowane, co świadczy o wysokiej czułości klasyfikacyjnej.

#### Panel 3: Eksperyment Antka (PC2, PC3, PC4 - Bez Albedo)
* **Charakterystyka:** Czysta inżynieria spektralna. Eliminacja składowej $PC1$ odcina informację o rzeźbie terenu i cieniach topograficznych (efekt płaskiego obrazu).
* **Wizualizacja terenu:** Ponieważ $PC2$ (odpowiadające za chlorofil) staje się składową dominującą (**41.60%**), całe otoczenie kopalni zalewa się intensywną zielenią wegetacyjną. Sama odkrywka zmienia barwę na kontrastujący granat i błękit. Pozwala to na bezpośrednie badanie zmian mineralogicznych dna wyrobiska bez zakłóceń wywołanych kątem padania promieni słonecznych.

---

### Główne Wnioski Badawcze
1.  **Poprawność Implementacji:** Tożsamość wyników autorskiej metody SVD z rozkładami referencyjnymi potwierdza stabilność matematyczną zaimplementowanego algorytmu potęgowego.
2.  **Korelacja ze Wskaźnikami:** Składowa $PC2$ wykazuje silną korelację przestrzenną ze wskaźnikiem roślinności **NDVI**, natomiast składowe $PC3$ i $PC4$ idealnie odwzorowują wskaźnik odkrytej ziemi **NDBI**.
3.  **Wartość Aplikacyjna:** Podczas gdy kompozycja SVD z $PC1-PC3$ stanowi idealne narzędzie do ogólnej kartografii i klasyfikacji pokrycia terenu, eksperymentalna kompozycja $PC2-PC4$ (bez albedo) posiada unikalne właściwości analityczne dla geologii strukturalnej i bio-monitoringu.

In [ ]:


import numpy as np
import matplotlib.pyplot as plt

TLO_GLOWNE = '#0B0F19'
TLO_KARTY = '#1E293B80'
KOLOR_TEKSTU = '#F1F5F9'
KOLOR_SIATKI = '#334155'
NEON_CYJAN = '#00F0FF'
NEON_FIOLET = '#8A2BE2'
NEON_ROZ = '#FF007F'
NEON_ZOLTY = '#FFD700' # Dodatkowy kolor dla PC4

plt.rcParams.update({
    "figure.facecolor": TLO_GLOWNE,
    "text.color": KOLOR_TEKSTU,
    "axes.labelcolor": '#94A3B8',
    "xtick.color": '#94A3B8',
    "ytick.color": '#94A3B8',
    "font.family": "sans-serif",
    "font.weight": "bold"
})

# --- IMPORTY I DANE ---
from pca_utils import wlasne_pca, pca_z_autorskim_svd
from data_utils import (
    stworz_macierz_3d_z_resamplingiem, 
    rozciagnij_na_2d, 
    przywroc_format_obrazka, 
    zaawansowana_normalizacja_kontrastu
)

folder = "C:/Users/Igor/Desktop/studia/sentinel2-pca/data/belchatow/2025-08-14-00_00_2025-08-14-23_59_Sentinel-2_L2A_"

#sciezki_wszystkie = [folder + "B02_(Raw).tiff", folder + "B03_(Raw).tiff", folder + "B04_(Raw).tiff", folder + "B05_(Raw).tiff", folder + "B08_(Raw).tiff", folder + "B8A_(Raw).tiff", folder + "B12_(Raw).tiff"]

sciezki_wszystkie = [folder + "B01_(Raw).tiff", folder + "B02_(Raw).tiff", folder + "B03_(Raw).tiff", folder + "B04_(Raw).tiff", folder + "B05_(Raw).tiff",
               folder + "B06_(Raw).tiff", folder + "B07_(Raw).tiff", folder + "B08_(Raw).tiff", folder + "B8A_(Raw).tiff", folder + "B09_(Raw).tiff", folder + "B12_(Raw).tiff"]

print("=== 1. OBLICZANIE DANYCH ===")
kostka = stworz_macierz_3d_z_resamplingiem(sciezki_wszystkie, [])
tabela_2d, wymiary = rozciagnij_na_2d(kostka)

# 1. Zwykłe PCA (Klasyka)
wynik_zwykle_2d, war_zwykle, total_zwykle = wlasne_pca(tabela_2d, liczba_skladowych=3)
procenty_zwykle = [(w / total_zwykle) * 100 for w in war_zwykle]
mapa_zwykle_rgb = zaawansowana_normalizacja_kontrastu(przywroc_format_obrazka(wynik_zwykle_2d, wymiary), 2, 98)

# 2. Autorskie SVD (PC1, PC2, PC3)
wynik_autorskie_2d, war_autorskie, total_autorskie = pca_z_autorskim_svd(tabela_2d, liczba_skladowych=3)
procenty_autorskie = [(w / total_autorskie) * 100 for w in war_autorskie]
mapa_autorskie_rgb = zaawansowana_normalizacja_kontrastu(przywroc_format_obrazka(wynik_autorskie_2d, wymiary), 2, 98)

# 3. POMYSŁ ANTKA: Autorskie SVD (bez PC1, bierzemy PC2, PC3, PC4)
print("-> Generowanie pomysłu Antka (SVD 4-składowe z odrzuceniem PC1)...")
# Pobieramy 4 składowe z naszego silnika
wynik_svd_4d, war_svd_4, total_svd_4 = pca_z_autorskim_svd(tabela_2d, liczba_skladowych=4)

# Trik Pythonowy: Wycinamy kolumny o indeksach 1, 2, 3 (czyli PC2, PC3, PC4)
wynik_bez_pc1_2d = wynik_svd_4d[:, 1:4]
procenty_bez_pc1 = [(w / total_svd_4) * 100 for w in war_svd_4[1:4]]

# Normalizujemy te "cichsze" składowe
mapa_bez_pc1_rgb = zaawansowana_normalizacja_kontrastu(przywroc_format_obrazka(wynik_bez_pc1_2d, wymiary), 2, 98)


print("=== 2. GENEROWANIE DASHBOARDU (TRÓJPODZIAŁ) ===")

# Poszerzamy figurę dla 3 kolumn
fig = plt.figure(figsize=(24, 10), layout="constrained")
fig.suptitle('Analiza Porównawcza Bełchatów\nZwykłe PCA vs. Autorskie SVD vs. SVD (Bez PC1)', 
             fontsize=26, color=KOLOR_TEKSTU)

ax_map1 = fig.add_subplot(2, 3, 1)
ax_map2 = fig.add_subplot(2, 3, 2)
ax_map3 = fig.add_subplot(2, 3, 3)

ax_bar1 = fig.add_subplot(2, 3, 4)
ax_bar2 = fig.add_subplot(2, 3, 5)
ax_bar3 = fig.add_subplot(2, 3, 6)

def zrob_karte(ax):
    ax.set_facecolor(TLO_KARTY)
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.tick_params(axis='both', which='both', length=0)

# ==========================================
# RZĄD 1: MAPY
# ==========================================
ax_map1.imshow(mapa_zwykle_rgb)
ax_map1.set_title("1. Zwykłe PCA\n(PC1, PC2, PC3)", color=KOLOR_TEKSTU, fontsize=15, pad=15)

ax_map2.imshow(mapa_autorskie_rgb)
ax_map2.set_title("2. Autorskie SVD z Z-score\n(PC1, PC2, PC3)", color=KOLOR_TEKSTU, fontsize=15, pad=15)

ax_map3.imshow(mapa_bez_pc1_rgb)
ax_map3.set_title("3. Eksperyment Antka\n(PC2, PC3, PC4 - Bez Albedo)", color=NEON_CYJAN, fontsize=15, pad=15)

for ax in [ax_map1, ax_map2, ax_map3]:
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_edgecolor('#ffffff30')
        spine.set_linewidth(1)

# ==========================================
# RZĄD 2: WYKRESY
# ==========================================
kolory_standard = [NEON_ROZ, NEON_CYJAN, NEON_FIOLET]
kolory_antka = [NEON_CYJAN, NEON_FIOLET, NEON_ZOLTY] # Odpowiadają PC2, PC3, PC4

def rysuj_wykres(ax, procenty, etykiety, kolory, tytul):
    zrob_karte(ax)
    slupki = ax.bar(etykiety, procenty, color=kolory, width=0.5, alpha=0.85)
    
    ax.set_title(tytul, color=KOLOR_TEKSTU, fontsize=14, pad=15)
    ax.set_ylabel('Wariancja (%)', fontsize=12)
    ax.set_ylim(0, max(60, max(procenty) + 10)) # Automatyczna wysokość dopasowana do PC1
    
    ax.yaxis.grid(True, linestyle='-', alpha=0.4, color=KOLOR_SIATKI)
    ax.xaxis.grid(False)
    
    for s in slupki:
        h = s.get_height()
        ax.annotate(f'{h:.2f}%',
                    xy=(s.get_x() + s.get_width() / 2, h),
                    xytext=(0, 10),
                    textcoords="offset points",
                    ha='center', va='bottom', fontsize=12, color=KOLOR_TEKSTU)

# Rysowanie wykresów z odpowiednimi etykietami
rysuj_wykres(ax_bar1, procenty_zwykle[:3], ['PC 1', 'PC 2', 'PC 3'], kolory_standard, f"Zwykłe PCA (Suma: {sum(procenty_zwykle[:3]):.2f}%)")
rysuj_wykres(ax_bar2, procenty_autorskie[:3], ['PC 1', 'PC 2', 'PC 3'], kolory_standard, f"Autorskie SVD (Suma: {sum(procenty_autorskie[:3]):.2f}%)")
rysuj_wykres(ax_bar3, procenty_bez_pc1, ['PC 2', 'PC 3', 'PC 4'], kolory_antka, f"Subtelne Detale (Suma PC2-4: {sum(procenty_bez_pc1):.2f}%)")

plt.savefig("dashboard_3_panele.png", dpi=300, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pca_utils import wlasne_pca
from data_utils import (
    stworz_macierz_3d_z_resamplingiem, 
    rozciagnij_na_2d, 
    przywroc_format_obrazka, 
    zaawansowana_normalizacja_kontrastu,
    wyswietl_mape
)

folder = "C:/Users/Igor/Desktop/studia/sentinel2-pca/data/teneryfa/2025-12-30-00_00_2025-12-30-23_59_Sentinel-2_L2A_"

sciezki_wszystkie = [folder + "B02_(Raw).tiff", folder + "B03_(Raw).tiff", folder + "B04_(Raw).tiff", folder + "B05_(Raw).tiff", folder + "B08_(Raw).tiff", folder + "B11_(Raw).tiff", folder + "B12_(Raw).tiff"]

#sciezki_wszystkie = [folder + "B01_(Raw).tiff", folder + "B02_(Raw).tiff", folder + "B03_(Raw).tiff", folder + "B04_(Raw).tiff", folder + "B05_(Raw).tiff",
#               folder + "B06_(Raw).tiff", folder + "B07_(Raw).tiff", folder + "B08_(Raw).tiff", folder + "B8A_(Raw).tiff", folder + "B09_(Raw).tiff",
#               folder + "B11_(Raw).tiff", folder + "B12_(Raw).tiff"]

print("=== START PROCESU ANNLITYCZNEGO ===")



kostka = stworz_macierz_3d_z_resamplingiem(sciezki_wszystkie, [])
tabela_2d, wymiary = rozciagnij_na_2d(kostka)
wynik_pca_2d, wariancje_skladowych, wariancja_total = wlasne_pca(tabela_2d, liczba_skladowych=3)
#wynik_pca_2d = wlasne_pca(tabela_2d, liczba_skladowych=3)
obraz_pca_3d = przywroc_format_obrazka(wynik_pca_2d, wymiary)
mapa_rgb = zaawansowana_normalizacja_kontrastu(obraz_pca_3d, dolny_percentyl=2, gorny_percentyl=98)

wyswietl_mape(mapa_rgb)



print("\n=== GENEROWANIE DOWODU NAUKOWEGO (SCREE PLOT) ===")

# 1. Obliczamy procenty Wyjaśnionej Wariancji (Explained Variance Ratio)
procenty = [(w / wariancja_total) * 100 for w in wariancje_skladowych]
suma_trzech = sum(procenty)

# 2. Rysowanie wykresu osypiska (Scree Plot)
plt.figure(figsize=(8, 5))
etykiety = [f'PC{i+1}' for i in range(len(procenty))]
slupki = plt.bar(etykiety, procenty, color=['#ff4c4c', '#4caf50', '#2196f3'])

# Dodawanie wartości procentowych nad słupkami
for slupek in slupki:
    wysokosc = slupek.get_height()
    plt.text(slupek.get_x() + slupek.get_width()/2., wysokosc + 1,
             f'{wysokosc:.2f}%', ha='center', va='bottom', fontweight='bold')

plt.title('Wykres Osypiska (Explained Variance Ratio)\nna podstawie Rozkładu Własnego', fontsize=14, fontweight='bold')
plt.ylabel('Wyjaśniona informacja (%)', fontsize=12)
plt.xlabel('Główne Składowe', fontsize=12)
plt.ylim(0, 100)
plt.grid(axis='y', linestyle='--', alpha=0.7)

print(f"-> 💡 WNIOSEK: Wasze 3 składowe RGB tłumaczą łącznie aż {suma_trzech:.2f}% całej geologii z 7 pasm!")
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pca_utils import wlasne_pca
from data_utils import (
    stworz_macierz_3d_z_resamplingiem, 
    rozciagnij_na_2d, 
    przywroc_format_obrazka, 
    zaawansowana_normalizacja_kontrastu,
    wyswietl_mape
)

folder = "C:/Users/Igor/Desktop/studia/sentinel2-pca/data/arabia/2026-05-15-00_00_2026-05-15-23_59_Sentinel-2_L2A_"

sciezki_wszystkie = [folder + "B02_(Raw).tiff", folder + "B03_(Raw).tiff", folder + "B04_(Raw).tiff", folder + "B05_(Raw).tiff", folder + "B08_(Raw).tiff", folder + "B11_(Raw).tiff", folder + "B12_(Raw).tiff"]

#sciezki_wszystkie = [folder + "B01_(Raw).tiff", folder + "B02_(Raw).tiff", folder + "B03_(Raw).tiff", folder + "B04_(Raw).tiff", folder + "B05_(Raw).tiff",
#               folder + "B06_(Raw).tiff", folder + "B07_(Raw).tiff", folder + "B08_(Raw).tiff", folder + "B8A_(Raw).tiff", folder + "B09_(Raw).tiff",
#               folder + "B11_(Raw).tiff", folder + "B12_(Raw).tiff"]

print("=== START PROCESU ANNLITYCZNEGO ===")



kostka = stworz_macierz_3d_z_resamplingiem(sciezki_wszystkie, [])
tabela_2d, wymiary = rozciagnij_na_2d(kostka)
wynik_pca_2d, wariancje_skladowych, wariancja_total = wlasne_pca(tabela_2d, liczba_skladowych=3)
#wynik_pca_2d = wlasne_pca(tabela_2d, liczba_skladowych=3)
obraz_pca_3d = przywroc_format_obrazka(wynik_pca_2d, wymiary)
mapa_rgb = zaawansowana_normalizacja_kontrastu(obraz_pca_3d, dolny_percentyl=2, gorny_percentyl=98)

wyswietl_mape(mapa_rgb)



print("\n=== GENEROWANIE DOWODU NAUKOWEGO (SCREE PLOT) ===")

# 1. Obliczamy procenty Wyjaśnionej Wariancji (Explained Variance Ratio)
procenty = [(w / wariancja_total) * 100 for w in wariancje_skladowych]
suma_trzech = sum(procenty)

# 2. Rysowanie wykresu osypiska (Scree Plot)
plt.figure(figsize=(8, 5))
etykiety = [f'PC{i+1}' for i in range(len(procenty))]
slupki = plt.bar(etykiety, procenty, color=['#ff4c4c', '#4caf50', '#2196f3'])

# Dodawanie wartości procentowych nad słupkami
for slupek in slupki:
    wysokosc = slupek.get_height()
    plt.text(slupek.get_x() + slupek.get_width()/2., wysokosc + 1,
             f'{wysokosc:.2f}%', ha='center', va='bottom', fontweight='bold')

plt.title('Wykres Osypiska (Explained Variance Ratio)\nna podstawie Rozkładu Własnego', fontsize=14, fontweight='bold')
plt.ylabel('Wyjaśniona informacja (%)', fontsize=12)
plt.xlabel('Główne Składowe', fontsize=12)
plt.ylim(0, 100)
plt.grid(axis='y', linestyle='--', alpha=0.7)

print(f"-> 💡 WNIOSEK: Wasze 3 składowe RGB tłumaczą łącznie aż {suma_trzech:.2f}% całej geologii z 7 pasm!")
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pca_utils import wlasne_pca
from data_utils import (
    stworz_macierz_3d_z_resamplingiem, 
    rozciagnij_na_2d, 
    przywroc_format_obrazka, 
    zaawansowana_normalizacja_kontrastu,
    wyswietl_mape
)

folder = "C:/Users/Igor/Desktop/studia/sentinel2-pca/data/almeria/2026-04-01-00_00_2026-04-01-23_59_Sentinel-2_L2A_"

sciezki_wszystkie = [folder + "B02_(Raw).tiff", folder + "B03_(Raw).tiff", folder + "B04_(Raw).tiff", folder + "B05_(Raw).tiff", folder + "B08_(Raw).tiff", folder + "B11_(Raw).tiff", folder + "B12_(Raw).tiff"]

#sciezki_wszystkie = [folder + "B01_(Raw).tiff", folder + "B02_(Raw).tiff", folder + "B03_(Raw).tiff", folder + "B04_(Raw).tiff", folder + "B05_(Raw).tiff",
#               folder + "B06_(Raw).tiff", folder + "B07_(Raw).tiff", folder + "B08_(Raw).tiff", folder + "B8A_(Raw).tiff", folder + "B09_(Raw).tiff",
#               folder + "B11_(Raw).tiff", folder + "B12_(Raw).tiff"]

print("=== START PROCESU ANNLITYCZNEGO ===")



kostka = stworz_macierz_3d_z_resamplingiem(sciezki_wszystkie, [])
tabela_2d, wymiary = rozciagnij_na_2d(kostka)
wynik_pca_2d, wariancje_skladowych, wariancja_total = wlasne_pca(tabela_2d, liczba_skladowych=3)
#wynik_pca_2d = wlasne_pca(tabela_2d, liczba_skladowych=3)
obraz_pca_3d = przywroc_format_obrazka(wynik_pca_2d, wymiary)
mapa_rgb = zaawansowana_normalizacja_kontrastu(obraz_pca_3d, dolny_percentyl=2, gorny_percentyl=98)

wyswietl_mape(mapa_rgb)



print("\n=== GENEROWANIE DOWODU NAUKOWEGO (SCREE PLOT) ===")

# 1. Obliczamy procenty Wyjaśnionej Wariancji (Explained Variance Ratio)
procenty = [(w / wariancja_total) * 100 for w in wariancje_skladowych]
suma_trzech = sum(procenty)

# 2. Rysowanie wykresu osypiska (Scree Plot)
plt.figure(figsize=(8, 5))
etykiety = [f'PC{i+1}' for i in range(len(procenty))]
slupki = plt.bar(etykiety, procenty, color=['#ff4c4c', '#4caf50', '#2196f3'])

# Dodawanie wartości procentowych nad słupkami
for slupek in slupki:
    wysokosc = slupek.get_height()
    plt.text(slupek.get_x() + slupek.get_width()/2., wysokosc + 1,
             f'{wysokosc:.2f}%', ha='center', va='bottom', fontweight='bold')

plt.title('Wykres Osypiska (Explained Variance Ratio)\nna podstawie Rozkładu Własnego', fontsize=14, fontweight='bold')
plt.ylabel('Wyjaśniona informacja (%)', fontsize=12)
plt.xlabel('Główne Składowe', fontsize=12)
plt.ylim(0, 100)
plt.grid(axis='y', linestyle='--', alpha=0.7)

print(f"-> 💡 WNIOSEK: Wasze 3 składowe RGB tłumaczą łącznie aż {suma_trzech:.2f}% całej geologii z 7 pasm!")
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pca_utils import wlasne_pca
from data_utils import (
    stworz_macierz_3d_z_resamplingiem, 
    rozciagnij_na_2d, 
    przywroc_format_obrazka, 
    zaawansowana_normalizacja_kontrastu,
    wyswietl_mape
)

folder = "C:/Users/Igor/Desktop/studia/sentinel2-pca/data/holandia/2026-05-01-00_00_2026-05-01-23_59_Sentinel-2_L2A_"

sciezki_wszystkie = [folder + "B02_(Raw).tiff", folder + "B03_(Raw).tiff", folder + "B04_(Raw).tiff", folder + "B05_(Raw).tiff", folder + "B08_(Raw).tiff", folder + "B11_(Raw).tiff", folder + "B12_(Raw).tiff"]

#sciezki_wszystkie = [folder + "B01_(Raw).tiff", folder + "B02_(Raw).tiff", folder + "B03_(Raw).tiff", folder + "B04_(Raw).tiff", folder + "B05_(Raw).tiff",
#               folder + "B06_(Raw).tiff", folder + "B07_(Raw).tiff", folder + "B08_(Raw).tiff", folder + "B8A_(Raw).tiff", folder + "B09_(Raw).tiff",
#               folder + "B11_(Raw).tiff", folder + "B12_(Raw).tiff"]

print("=== START PROCESU ANNLITYCZNEGO ===")



kostka = stworz_macierz_3d_z_resamplingiem(sciezki_wszystkie, [])
tabela_2d, wymiary = rozciagnij_na_2d(kostka)
wynik_pca_2d, wariancje_skladowych, wariancja_total = wlasne_pca(tabela_2d, liczba_skladowych=3)
#wynik_pca_2d = wlasne_pca(tabela_2d, liczba_skladowych=3)
obraz_pca_3d = przywroc_format_obrazka(wynik_pca_2d, wymiary)
mapa_rgb = zaawansowana_normalizacja_kontrastu(obraz_pca_3d, dolny_percentyl=2, gorny_percentyl=98)

wyswietl_mape(mapa_rgb)



print("\n=== GENEROWANIE DOWODU NAUKOWEGO (SCREE PLOT) ===")

# 1. Obliczamy procenty Wyjaśnionej Wariancji (Explained Variance Ratio)
procenty = [(w / wariancja_total) * 100 for w in wariancje_skladowych]
suma_trzech = sum(procenty)

# 2. Rysowanie wykresu osypiska (Scree Plot)
plt.figure(figsize=(8, 5))
etykiety = [f'PC{i+1}' for i in range(len(procenty))]
slupki = plt.bar(etykiety, procenty, color=['#ff4c4c', '#4caf50', '#2196f3'])

# Dodawanie wartości procentowych nad słupkami
for slupek in slupki:
    wysokosc = slupek.get_height()
    plt.text(slupek.get_x() + slupek.get_width()/2., wysokosc + 1,
             f'{wysokosc:.2f}%', ha='center', va='bottom', fontweight='bold')

plt.title('Wykres Osypiska (Explained Variance Ratio)\nna podstawie Rozkładu Własnego', fontsize=14, fontweight='bold')
plt.ylabel('Wyjaśniona informacja (%)', fontsize=12)
plt.xlabel('Główne Składowe', fontsize=12)
plt.ylim(0, 100)
plt.grid(axis='y', linestyle='--', alpha=0.7)

print(f"-> 💡 WNIOSEK: Wasze 3 składowe RGB tłumaczą łącznie aż {suma_trzech:.2f}% całej geologii z 7 pasm!")
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pca_utils import wlasne_pca
from data_utils import (
    stworz_macierz_3d_z_resamplingiem, 
    rozciagnij_na_2d, 
    przywroc_format_obrazka, 
    zaawansowana_normalizacja_kontrastu,
    wyswietl_mape
)

folder = "C:/Users/Igor/Desktop/studia/sentinel2-pca/data/andaluzja/2026-05-15-00_00_2026-05-15-23_59_Sentinel-2_L2A_"

sciezki_wszystkie = [folder + "B02_(Raw).tiff", folder + "B03_(Raw).tiff", folder + "B04_(Raw).tiff", folder + "B05_(Raw).tiff", folder + "B08_(Raw).tiff", folder + "B11_(Raw).tiff", folder + "B12_(Raw).tiff"]

#sciezki_wszystkie = [folder + "B01_(Raw).tiff", folder + "B02_(Raw).tiff", folder + "B03_(Raw).tiff", folder + "B04_(Raw).tiff", folder + "B05_(Raw).tiff",
#               folder + "B06_(Raw).tiff", folder + "B07_(Raw).tiff", folder + "B08_(Raw).tiff", folder + "B8A_(Raw).tiff", folder + "B09_(Raw).tiff",
#               folder + "B11_(Raw).tiff", folder + "B12_(Raw).tiff"]

print("=== START PROCESU ANNLITYCZNEGO ===")



kostka = stworz_macierz_3d_z_resamplingiem(sciezki_wszystkie, [])
tabela_2d, wymiary = rozciagnij_na_2d(kostka)
wynik_pca_2d, wariancje_skladowych, wariancja_total = wlasne_pca(tabela_2d, liczba_skladowych=3)
#wynik_pca_2d = wlasne_pca(tabela_2d, liczba_skladowych=3)
obraz_pca_3d = przywroc_format_obrazka(wynik_pca_2d, wymiary)
mapa_rgb = zaawansowana_normalizacja_kontrastu(obraz_pca_3d, dolny_percentyl=2, gorny_percentyl=98)

wyswietl_mape(mapa_rgb)



print("\n=== GENEROWANIE DOWODU NAUKOWEGO (SCREE PLOT) ===")

# 1. Obliczamy procenty Wyjaśnionej Wariancji (Explained Variance Ratio)
procenty = [(w / wariancja_total) * 100 for w in wariancje_skladowych]
suma_trzech = sum(procenty)

# 2. Rysowanie wykresu osypiska (Scree Plot)
plt.figure(figsize=(8, 5))
etykiety = [f'PC{i+1}' for i in range(len(procenty))]
slupki = plt.bar(etykiety, procenty, color=['#ff4c4c', '#4caf50', '#2196f3'])

# Dodawanie wartości procentowych nad słupkami
for slupek in slupki:
    wysokosc = slupek.get_height()
    plt.text(slupek.get_x() + slupek.get_width()/2., wysokosc + 1,
             f'{wysokosc:.2f}%', ha='center', va='bottom', fontweight='bold')

plt.title('Wykres Osypiska (Explained Variance Ratio)\nna podstawie Rozkładu Własnego', fontsize=14, fontweight='bold')
plt.ylabel('Wyjaśniona informacja (%)', fontsize=12)
plt.xlabel('Główne Składowe', fontsize=12)
plt.ylim(0, 100)
plt.grid(axis='y', linestyle='--', alpha=0.7)

print(f"-> 💡 WNIOSEK: Wasze 3 składowe RGB tłumaczą łącznie aż {suma_trzech:.2f}% całej geologii z 7 pasm!")
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pca_utils import wlasne_pca
from data_utils import (
    stworz_macierz_3d_z_resamplingiem, 
    rozciagnij_na_2d, 
    przywroc_format_obrazka, 
    zaawansowana_normalizacja_kontrastu,
    wyswietl_mape
)

folder = "C:/Users/Igor/Desktop/studia/sentinel2-pca/data/sahara/2026-05-15-00_00_2026-05-15-23_59_Sentinel-2_L2A_"

#sciezki_wszystkie = [folder + "B02_(Raw).tiff", folder + "B03_(Raw).tiff", folder + "B04_(Raw).tiff", folder + "B05_(Raw).tiff", folder + "B08_(Raw).tiff", folder + "B11_(Raw).tiff", folder + "B12_(Raw).tiff"]

sciezki_wszystkie = [folder + "B01_(Raw).tiff", folder + "B02_(Raw).tiff", folder + "B03_(Raw).tiff", folder + "B04_(Raw).tiff", folder + "B05_(Raw).tiff",
               folder + "B06_(Raw).tiff", folder + "B07_(Raw).tiff", folder + "B08_(Raw).tiff", folder + "B8A_(Raw).tiff", folder + "B09_(Raw).tiff",
               folder + "B11_(Raw).tiff", folder + "B12_(Raw).tiff"]

print("=== START PROCESU ANNLITYCZNEGO ===")



kostka = stworz_macierz_3d_z_resamplingiem(sciezki_wszystkie, [])
tabela_2d, wymiary = rozciagnij_na_2d(kostka)
wynik_pca_2d, wariancje_skladowych, wariancja_total = wlasne_pca(tabela_2d, liczba_skladowych=3)
#wynik_pca_2d = wlasne_pca(tabela_2d, liczba_skladowych=3)
obraz_pca_3d = przywroc_format_obrazka(wynik_pca_2d, wymiary)
mapa_rgb = zaawansowana_normalizacja_kontrastu(obraz_pca_3d, dolny_percentyl=2, gorny_percentyl=98)

wyswietl_mape(mapa_rgb)



print("\n=== GENEROWANIE DOWODU NAUKOWEGO (SCREE PLOT) ===")

# 1. Obliczamy procenty Wyjaśnionej Wariancji (Explained Variance Ratio)
procenty = [(w / wariancja_total) * 100 for w in wariancje_skladowych]
suma_trzech = sum(procenty)

# 2. Rysowanie wykresu osypiska (Scree Plot)
plt.figure(figsize=(8, 5))
etykiety = [f'PC{i+1}' for i in range(len(procenty))]
slupki = plt.bar(etykiety, procenty, color=['#ff4c4c', '#4caf50', '#2196f3'])

# Dodawanie wartości procentowych nad słupkami
for slupek in slupki:
    wysokosc = slupek.get_height()
    plt.text(slupek.get_x() + slupek.get_width()/2., wysokosc + 1,
             f'{wysokosc:.2f}%', ha='center', va='bottom', fontweight='bold')

plt.title('Wykres Osypiska (Explained Variance Ratio)\nna podstawie Rozkładu Własnego', fontsize=14, fontweight='bold')
plt.ylabel('Wyjaśniona informacja (%)', fontsize=12)
plt.xlabel('Główne Składowe', fontsize=12)
plt.ylim(0, 100)
plt.grid(axis='y', linestyle='--', alpha=0.7)

print(f"-> 💡 WNIOSEK: Wasze 3 składowe RGB tłumaczą łącznie aż {suma_trzech:.2f}% całej geologii z 7 pasm!")
plt.show()